# 01. Generative AI: Hugging Face Transformers

## Algorithm Category
**Type**: Generative AI - Language Models  
**Complexity**: Medium  
**Use Case**: Text generation, classification, question answering, and NLP tasks

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand the Hugging Face Transformers library and its ecosystem
- Use the Pipeline API for common NLP tasks
- Load and use pre-trained models with AutoModel and AutoTokenizer
- Generate text with various sampling strategies
- Perform text classification and question answering
- Optimize models for GPU/CPU inference
- Understand tokenization and model architecture basics

## Historical Context

Hugging Face was founded in 2016 and quickly became the de facto standard for transformer models in NLP. The Transformers library democratized access to state-of-the-art models like BERT, GPT, T5, and many others.

## When to Use Hugging Face Transformers

Hugging Face Transformers is appropriate when:
- You need quick access to pre-trained models for NLP tasks
- You want to experiment with different models without complex setup
- You need text generation, classification, or question answering
- You want to fine-tune models on your own data


## Theory & Mechanics

### What is Hugging Face Transformers?

Hugging Face Transformers is an open-source library that provides easy access to thousands of pre-trained machine learning models for natural language processing, computer vision, and audio tasks. Built on PyTorch and TensorFlow, it offers a unified API to load, fine-tune, and deploy state-of-the-art models.

### Key Concepts

1. **Pipelines**: High-level abstractions for common tasks (text-generation, classification, QA)
2. **AutoModel/AutoTokenizer**: Automatic model and tokenizer loading based on model name
3. **Tokenization**: Converting text to numeric IDs that models can process
4. **Model Architecture**: Transformer-based models using self-attention mechanisms

### How It Works

1. **Input Processing**: Text is tokenized into token IDs
2. **Model Forward Pass**: Token IDs pass through transformer layers
3. **Output Generation**: Model produces logits (raw scores) for each token
4. **Decoding**: Logits are converted back to text tokens


## Installation & Setup


In [ ]:
# ============================================
# INSTALLATION: Setting Up Hugging Face Transformers
# ============================================

# Hugging Face Transformers is a library for using pre-trained transformer models
# It provides easy access to thousands of state-of-the-art NLP models
# Models like GPT, BERT, T5, etc. are available with just a few lines of code

# Install transformers if not already installed
# Uncomment the line below if you need to install:
# !pip install transformers torch
# Note: torch (PyTorch) is required for running transformer models

# Import transformers library
import transformers  # Main Hugging Face Transformers library

# Check version
print(f"Transformers version: {transformers.__version__}")
# Different versions may have different features/APIs

Transformers version: 4.57.3


## Implementation


In [ ]:
# ============================================
# SETTING UP PYTHON PATH: For Custom Imports
# ============================================

# Add project root to Python path so we can import our custom modules
import sys  # System-specific parameters and functions
from pathlib import Path  # Object-oriented filesystem paths

# Get the project root directory
# Path().resolve() gets current directory (where notebook is)
# .parent.parent goes up two levels (notebooks/generative_ai -> notebooks -> project root)
project_root = Path().resolve().parent.parent

# Add project root to sys.path if not already there
# This allows Python to find modules in the src/ directory
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
# sys.path is a list of directories Python searches for modules
# insert(0, ...) adds to the beginning (highest priority)

# ============================================
# IMPORTING LIBRARIES: Transformers and Utilities
# ============================================

# Hugging Face Transformers library
from transformers import (
    pipeline,  # High-level API for common NLP tasks (text generation, classification, etc.)
    AutoTokenizer,  # Automatically load the correct tokenizer for a model
    AutoModelForCausalLM  # Automatically load causal language models (like GPT)
)
# pipeline: Simplest way to use models (one line of code)
# AutoTokenizer: Converts text to token IDs (numbers models understand)
# AutoModelForCausalLM: Language models that generate text (GPT-style)

# Our custom utility functions (organized in src/llm/ directory)
from src.llm.transformers_utils import (
    load_transformers_model,  # Load model and tokenizer together
    generate_text,  # Generate text with various options
    classify_text,  # Classify text using transformer models
    get_model_info  # Get information about a model (size, parameters, etc.)
)

# PyTorch: Deep learning framework (required by transformers)
import torch  # PyTorch library for tensor operations and neural networks

print("Libraries imported successfully!")  # Confirm all imports worked

Libraries imported successfully!


In [ ]:
# ============================================
# EXAMPLE 1: Using Pipeline API (Simplest Approach)
# ============================================

# pipeline() is the easiest way to use transformer models
# It handles model loading, tokenization, and generation automatically
# Just specify the task and model name!

# Create a text generation pipeline
# "text-generation": Task type (generating text)
# model="distilgpt2": Model to use (DistilGPT-2 is a smaller, faster version of GPT-2)
#   - GPT-2: Generative Pre-trained Transformer 2 (OpenAI's language model)
#   - DistilGPT-2: Distilled (compressed) version, faster but slightly less capable
generator = pipeline("text-generation", model="distilgpt2")
# This automatically downloads the model on first use (may take a minute)

# ============================================
# GENERATING TEXT: From a Prompt
# ============================================

# Define a prompt (starting text)
prompt = "Once upon a time, in a world of machines,"
# The model will continue this text

# Generate text
# generator() takes the prompt and generates continuation
# max_length=50: Maximum total length (prompt + generated text)
# num_return_sequences=1: Number of different completions to generate
result = generator(prompt, max_length=50, num_return_sequences=1)
# Returns list of dictionaries, each with "generated_text" key

print("Generated text:")
print(result[0]["generated_text"])  # Display the generated text
# result[0] is the first (and only) generated sequence
# ["generated_text"] gets the actual text string

# Interpretation:
# - Model continues the prompt in a coherent way
# - Generated text should make sense in context
# - Quality depends on the model and prompt
# - Different runs may produce different outputs (non-deterministic)

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use cpu
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generated text:
Once upon a time, in a world of machines, a single human has a great deal to offer. He could make his money selling his technology to people like him.


“I“m not going to get into that.“


In [ ]:
# ============================================
# EXAMPLE 2: Using Helper Functions (More Control)
# ============================================

# load_transformers_model() loads model and tokenizer separately
# This gives more control over the generation process
# "gpt2": Full GPT-2 model (larger than DistilGPT-2, better quality)
# task="text-generation": Specify the task
model, tokenizer = load_transformers_model("gpt2", task="text-generation")
# Returns:
# - model: The actual neural network (PyTorch model)
# - tokenizer: Converts text to token IDs and back

# ============================================
# GETTING MODEL INFORMATION
# ============================================

# get_model_info() retrieves information about a model
# This helps understand model capabilities and requirements
model_info = get_model_info("gpt2")
# Returns dictionary with model metadata

print("Model Information:")
# Loop through and display all information
for key, value in model_info.items():
    print(f"  {key}: {value}")
# May include: model size, number of parameters, architecture, etc.

# Interpretation:
# - Model size: Larger models = better quality but slower
# - Parameters: Number of learnable weights (GPT-2 has ~124M parameters)
# - Architecture: Transformer-based (self-attention mechanism)
# - This information helps choose the right model for your needs

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model Information:
  model_name: gpt2
  model_type: gpt2
  vocab_size: 50257
  max_position_embeddings: 1024
  hidden_size: 768
  num_attention_heads: 12
  num_layers: 12
  tokenizer_vocab_size: 50257
  pad_token: None
  eos_token: <|endoftext|>
  bos_token: <|endoftext|>


In [6]:
# Example 3: Text generation with custom parameters
prompt = "AI can help healthcare by"
generated = generate_text(
    model, tokenizer, prompt,
    max_new_tokens=50,
    temperature=0.7,
    top_p=0.9,
    do_sample=True
)
print("Generated text:")
print(generated[0])

Generated text:
 offering a way to keep patients safe and healthy through a range of services, including medication and diagnostic tests.

"The government has been working closely with our healthcare providers and other stakeholders to ensure the quality and safety of the healthcare system is maintained,"


## Validation & Testing

Let's validate our model outputs and test different configurations.


In [7]:
# Validation: Check that generated text is not empty
assert len(generated) > 0, "No text generated!"
assert len(generated[0]) > 0, "Generated text is empty!"
assert isinstance(generated[0], str), "Generated text should be a string"
print("✓ Text generation validation passed")

✓ Text generation validation passed


## Performance Benchmarking


In [8]:
import time

# Benchmark generation speed
prompts = ["The future of AI is", "Machine learning enables", "Deep learning models"]
times = []

for prompt in prompts:
    start = time.time()
    _ = generate_text(model, tokenizer, prompt, max_new_tokens=20)
    elapsed = time.time() - start
    times.append(elapsed)

avg_time = sum(times) / len(times)
print(f"Average generation time: {avg_time:.3f} seconds")
print(f"Tokens per second: {20/avg_time:.1f}")

Average generation time: 0.982 seconds
Tokens per second: 20.4


## Summary & Key Takeaways

- Hugging Face Transformers provides easy access to pre-trained models
- Pipeline API is great for quick prototyping
- AutoModel/AutoTokenizer give more control for custom use cases
- Tokenization is crucial for proper model input
- GPU acceleration significantly speeds up inference
